# 05 · Baseline 2: RAG (BM25, dense, hybrid)

RAG = **R**etrieve a few relevant records, then let the LLM **G**enerate the answer from them (here k = 20).
Each record (one order line) is one chunk, so retrieval never cuts a record in half.

| Retriever | How it ranks records | Good at |
|---|---|---|
| **BM25** | keyword overlap, rare words count more | exact ids like `O52526` |
| **Dense** | cosine similarity of embeddings (`multilingual-e5-small`) | meaning, across languages |
| **Hybrid** | reciprocal rank fusion of both lists | a mix of the two |

In [1]:
import json
from pathlib import Path

import pandas as pd
from rlm_logistics.llm import LLM
from rlm_logistics import methods

ROOT = Path("..")
DEV = ROOT / "results/dev"          # dev runs are not committed; final runs (Phase 5) go to results/runs
meta = json.loads((ROOT / "data/processed/contexts.json").read_text())
qs = pd.read_json(ROOT / "data/processed/questions.jsonl", lines=True)
lines = (ROOT / "data/processed/context_2m.txt").read_text(encoding="utf-8").split("\n")
sample = qs[qs["size"] == "32k"].groupby("tier").head(1)   # one question per tier: a smoke test
sample[["id", "question"]]

,id,question
0,32k-T1-0,What is the main complaint in the review of or...
6,32k-T2-0,Among orders delivered after the estimated dat...
12,32k-T3-0,"Among all orders, list every pair of orders fr..."


In [2]:
from rlm_logistics.rag import BM25, Dense, Hybrid

bm25 = BM25(lines)
dense = Dense(lines, cache=ROOT / "data/processed/embeddings_e5small.npy")  # ~7 min the first time, then cached
hybrid = Hybrid(bm25, dense)
retrievers = {"rag_bm25": bm25, "rag_dense": dense, "rag_hybrid": hybrid}

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 1. Retrieval quality on lookups (T1)
For a T1 question the right record is known: the order named in the question.
**Recall@20** = how often that record is among the 20 retrieved. If it is missing, the LLM cannot answer.

In [3]:
t1 = qs[qs.tier == "T1"]
target = t1.question.str.extract(r"(O\d{5})")[0]
recall = {name: sum(t in " ".join(lines[i][:15] for i in r.search(q, 20, meta[s]["n_records"]))
                    for q, s, t in zip(t1.question, t1["size"], target)) / len(t1)
          for name, r in retrievers.items()}
pd.Series(recall, name="T1 recall@20")

rag_bm25      1.0
rag_dense     0.1
rag_hybrid    0.8
Name: T1 recall@20, dtype: float64

**Why dense fails on lookups:** an embedding captures *meaning*, and an order id has none.
`O52526` and `O52527` look almost identical to the model. BM25 treats the id as a rare word and finds it at once.

**Why hybrid sits in between:** rank fusion adds 1/(60 + rank) from each list. A record that is #1 in BM25 but missing from
the dense top 100 scores 1/60 ≈ 0.017, while a record ranked ~5th in *both* lists scores 2/65 ≈ 0.031 and beats it.

## 2. The built-in ceiling on aggregation (T2) and pairwise (T3)
A T2 question about late orders needs *every* late order in the context. RAG passes 20 records, whatever the context size.

In [4]:
for size, m in meta.items():
    ctx = pd.read_csv(ROOT / "data/processed/records.csv").iloc[:m["n_records"]]
    print(f"{size:>5}: {(ctx.days_late > 0).sum():>5} late orders in the context, RAG shows the LLM 20 records")

  32k:    29 late orders in the context, RAG shows the LLM 20 records
 128k:    97 late orders in the context, RAG shows the LLM 20 records


 512k:   456 late orders in the context, RAG shows the LLM 20 records
   2m:  1712 late orders in the context, RAG shows the LLM 20 records


## 3. Smoke test on the dev model
One question per tier at 32K, for each retriever (9 calls, ~1 minute each on CPU).

In [5]:
llm = LLM.from_config("dev")
n = meta["32k"]["n_records"]
rows = [methods.run(name, q, lambda: methods.rag(q["question"], llm, lines, r, n), llm.model, DEV)
        for name, r in retrievers.items() for q in sample.to_dict("records")]
pd.DataFrame(rows).pivot(index="method", columns="tier", values="score")

tier,T1,T2,T3
method,,,
rag_bm25,1.0,0.0,0.000000
rag_dense,0.0,0.0,0.060606
rag_hybrid,0.0,0.0,0.031746


In [6]:
pd.DataFrame(rows).groupby("method")[["input_tokens", "seconds"]].mean().round(0)

,input_tokens,seconds
method,,
rag_bm25,2457.0,165.0
rag_dense,2429.0,76.0
rag_hybrid,2461.0,146.0


## What to notice
* RAG prompts are ~3K tokens instead of ~32K: about 10× cheaper than direct LLM, at every context size.
* On T1, success depends on retrieval first: BM25 can find the order, dense usually cannot.
* On T2/T3, even perfect retrieval cannot help: 20 records are not "all late orders". This is the gap RLM is meant to close.